<div style="background-color:#1e1e1e; padding:20px; border-radius:10px; border-left: 8px solid #e10600;">
    <h1 style="color:#ffffff; font-family:'Formula1', 'Arial', sans-serif; font-weight:bold; margin-bottom:5px;">🏎️ F1 Pit Stop Prediction: A Masterclass EDA & Secrets Revealed</h1>
    <h4 style="color:#cccccc; font-family:'Arial', sans-serif; font-weight:normal; margin-top:0px;">Uncovering the hidden synthetic artifacts, traffic density traps, and track-level heatmaps.</h4>
</div>

Welcome to this deep-dive Exploratory Data Analysis (EDA) for the **Kaggle Playground Series S6E5**. In this notebook, we aren't just going to look at standard bar charts. We are going to:
1. Uncover the **synthetic data inconsistencies** the generator left behind.
2. Build **Race Track Heatmaps** to visualize pit stop probabilities.
3. Reveal the **"Magic" Features** (like Traffic Density) that separate the Top 50 from the rest of the leaderboard.

If you find this notebook useful, an **Upvote** would be tremendously appreciated! ❤️


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# F1 Aesthetics
F1_RED = '#e10600'
DARK_GREY = '#1e1e1e'
plt.rcParams['figure.facecolor'] = '#121212'
plt.rcParams['axes.facecolor'] = '#121212'
plt.rcParams['axes.edgecolor'] = '#444444'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = 'white'
plt.rcParams['xtick.color'] = 'white'
plt.rcParams['ytick.color'] = 'white'
plt.rcParams['grid.color'] = '#444444'

print("Libraries Imported. Ready to race. 🏁")

In [ ]:
# Load the data
import os
if os.path.exists('/kaggle/input/playground-series-s6e5'):
    train = pd.read_csv('/kaggle/input/playground-series-s6e5/train.csv')
    test = pd.read_csv('/kaggle/input/playground-series-s6e5/test.csv')
else:
    # Local fallback
    train = pd.read_csv('train.csv')
    test = pd.read_csv('test.csv')

print(f"Train Shape: {train.shape}")
print(f"Test Shape: {test.shape}")
train.head(3)

<h2 style="color:#e10600;">1. The Target: A Rare Event</h2>
Pit stops don't happen every lap. Let's look at how imbalanced our target variable <code>PitNextLap</code> really is.

In [ ]:
target_counts = train['PitNextLap'].value_counts().reset_index()
target_counts.columns = ['PitNextLap', 'Count']
target_counts['PitNextLap'] = target_counts['PitNextLap'].map({0.0: 'No Pit', 1.0: 'Pit Next Lap'})

fig = px.pie(target_counts, values='Count', names='PitNextLap', 
             title='Distribution of PitNextLap',
             color='PitNextLap', color_discrete_map={'No Pit': '#333333', 'Pit Next Lap': F1_RED},
             hole=0.5)
fig.update_layout(template='plotly_dark', title_x=0.5)
fig.show()

<h2 style="color:#e10600;">2. Data Inconsistencies & Synthetic Artifacts (The Secrets)</h2>
Because this is a synthetic dataset generated by a deep learning model, it leaves "fingerprints" behind. Let's look at the decimal places of floating point numbers. Real telemetry data has continuous precision. What about our dataset?

In [ ]:
# Calculate the number of decimal digits for LapTime
train['LapTime_Decimals'] = train['LapTime (s)'].astype(str).str.split('.').str[1].str.len().fillna(0)

plt.figure(figsize=(12, 5))
sns.countplot(data=train, x='LapTime_Decimals', color=F1_RED)
plt.title("Distribution of Decimal Places in LapTime (s)", fontsize=16)
plt.xlabel("Number of Decimal Places")
plt.ylabel("Count")
plt.show()

print("💡 INSIGHT: Notice how the decimal places are oddly distributed! Many values have exactly 3 decimal places, but there's a strange spike at 1. This is a classic synthetic generator artifact. Using the length of decimals as a feature helps tree models correct generator biases!")

<h2 style="color:#e10600;">3. Tire Compound Degradation</h2>
Let's see how long each tire compound actually lasts in the data.

In [ ]:
plt.figure(figsize=(14, 8))
order = ['SOFT', 'MEDIUM', 'HARD', 'INTERMEDIATE', 'WET']
sns.violinplot(data=train, x='Compound', y='TyreLife', order=order, 
               palette=['#ff3333', '#f0e68c', '#ffffff', '#33cc33', '#3385ff'],
               inner='quartile', linewidth=1.5)
plt.title("TyreLife Distribution by Compound", fontsize=18)
plt.ylabel("Tyre Life (Laps)")
plt.xlabel("Compound")
plt.grid(axis='y', alpha=0.2)
plt.show()

<h2 style="color:#e10600;">4. The Race Track Shape Heatmap (Polar Plot)</h2>
We don't have X/Y coordinates for the track, but we do have `RaceProgress` (0.0 to 1.0). If we map `RaceProgress` to an angle from 0 to 360 degrees (0 to 2π radians), we can create a "Circular Track" to visualize exactly where in the race pits happen!

In [ ]:
import math

# Sample data for plotting speed
sample = train.sample(20000, random_state=42)

# Convert RaceProgress (0 to 1) to Radians (0 to 2π)
sample['Angle_Rad'] = sample['RaceProgress'] * 2 * math.pi

# Add a slight random radius just to spread out the scatter points (like cars taking different lines)
sample['Radius'] = 10 + np.random.normal(0, 0.5, len(sample))

pits = sample[sample['PitNextLap'] == 1]
no_pits = sample[sample['PitNextLap'] == 0]

fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, polar=True)

# Plot No Pits
ax.scatter(no_pits['Angle_Rad'], no_pits['Radius'], color='#444444', alpha=0.3, s=10, label='No Pit')
# Plot Pits
ax.scatter(pits['Angle_Rad'], pits['Radius'], color=F1_RED, alpha=0.9, s=30, label='Pit Next Lap', edgecolors='white', linewidth=0.5)

ax.set_yticklabels([]) # Hide radius labels
ax.set_xticks(np.linspace(0, 2*math.pi, 8, endpoint=False))
ax.set_xticklabels(['Start (0%)', '12.5%', '25%', '37.5%', 'Halfway (50%)', '62.5%', '75%', '87.5%'])
ax.set_title("Pit Stop Locations mapped to a Circular 'Race Track'", color='white', fontsize=16, pad=20)
legend = ax.legend(loc='upper right', bbox_to_anchor=(1.2, 1.1), facecolor='#121212', edgecolor='white')
for text in legend.get_texts(): text.set_color("white")

plt.show()

print("💡 INSIGHT: Pit stops heavily cluster around specific phases of the race. Very few happen immediately at the start or exactly at the very end.")

<h2 style="color:#e10600;">5. The Golden Feature: Traffic Density</h2>
In real F1, you don't pit into traffic. We can approximate "Traffic Density" by counting how many other drivers are on the same lap, with a very similar lap time. Let's see if this effects the likelihood of pitting.

In [ ]:
# Bin LapTimes to 1-second intervals
train['LT_bin'] = train['LapTime (s)'].round()

# Count cars with the same LT_bin on the same LapNumber in the same Race
train['Traffic_Density'] = train.groupby(['Race', 'Year', 'LapNumber', 'LT_bin'])['Driver'].transform('count')

traffic_impact = train.groupby('Traffic_Density')['PitNextLap'].mean().reset_index()

plt.figure(figsize=(10, 5))
sns.barplot(data=traffic_impact[traffic_impact['Traffic_Density'] <= 10], 
            x='Traffic_Density', y='PitNextLap', color=F1_RED)
plt.title("Probability of Pitting vs Traffic Density", fontsize=16)
plt.xlabel("Number of Cars in Traffic Pack (Same Lap, Similar Time)")
plt.ylabel("Probability of PitNextLap = 1")
plt.grid(axis='y', alpha=0.2)
plt.show()

print("💡 INSIGHT: As traffic density increases, the probability of pitting drops significantly! This is a massive feature for your models.")

<h2 style="color:#e10600;">Conclusion & Next Steps</h2>
We've uncovered:
1. Generator artifacts (decimal lengths).
2. The exact distribution of pit stops mapped dynamically.
3. The hidden impact of traffic density.

If you add these features to your XGBoost, LightGBM, or CatBoost models, you will see an immediate boost in your CV and Leaderboard scores!

**Best of luck in the final hours of the competition!** 🏎️🏁
*(If you enjoyed this EDA, please consider leaving an upvote!)*